# 🫀 CardioIA — Fase 2 | Parte 1: Extração de Informações e Ontologia Clínica
### Diagnóstico Automatizado: O Estetoscópio Digital do Século XXI

**Autores:** Gabriela de Andrade Alves (RM567740) e Leonardo de Mattos Oliveira (RM568219)  
**Instituição:** FIAP — Inteligência Artificial 2025  
**Metodologia:** Problem Based Learning (PBL)

---

## 🎯 1. Contextualização e Objetivos

Nesta etapa do projeto CardioIA, implementamos um sistema de **Processamento de Linguagem Natural (NLP)** e **Inferência Baseada em Ontologia** para simular o processo cognitivo de triagem médica.

### O Desafio Clínico
Estudos epidemiológicos apontam que cerca de 80% das informações médicas em hospitais residem em texto não estruturado (anamneses, queixas principais e anotações de evolução). Transformar relatos livres de pacientes em hipóteses diagnósticas estruturadas reduz o tempo porta-balão em emergências e apoia a decisão de médicos e enfermeiros.

### Entregáveis Contemplados
1. Leitura e interpretação de **10 frases completas** de pacientes contendo: sintoma sentido, tempo de início e impacto funcional.
2. Mapa de Conhecimento clínico estruturado em `.csv` (`sintoma_1`, `sintoma_2`, `doenca_associada`).
3. Algoritmo de extração de entidades e regras de associação com níveis de confiança (**Alta** vs **Moderada**).


## 📦 2. Importação das Dependências


In [ ]:
import os
import re
import csv
import pandas as pd
from collections import defaultdict

# Configurações de caminhos relativos robustos
BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")

ARQUIVO_FRASES = os.path.join(DATA_DIR, "sintomas_pacientes.txt")
ARQUIVO_MAPA = os.path.join(DATA_DIR, "mapa_conhecimento.csv")

print(f"Diretório de dados: {DATA_DIR}")
print(f"Arquivo de frases existe: {os.path.exists(ARQUIVO_FRASES)}")
print(f"Arquivo do mapa existe:   {os.path.exists(ARQUIVO_MAPA)}")


## 📋 3. Carregamento e Inspeção dos Relatos dos Pacientes
As 10 frases foram elaboradas simulando relatos de pacientes reais, contendo:
- **O que sente:** dor no peito, falta de ar, inchaço, palpitações, etc.
- **Quando começou:** há dois dias, desde ontem à noite, há mais de uma semana, etc.
- **Impacto na rotina:** piora ao subir escadas, sufocamento ao deitar, impede caminhar até a padaria, etc.


In [ ]:
# Carregando as frases do arquivo .txt
frases_pacientes = []
with open(ARQUIVO_FRASES, "r", encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if linha:
            frases_pacientes.append(linha)

df_pacientes = pd.DataFrame({
    "Paciente": [f"Paciente #{i:02d}" for i in range(1, len(frases_pacientes) + 1)],
    "Relato Clínico": frases_pacientes
})

pd.set_option('display.max_colwidth', None)
df_pacientes


## 🗺️ 4. Mapa de Conhecimento Clínico (Ontologia de Sintomas)
O mapa associa combinações sintomáticas a possíveis doenças cardiovasculares e correlatas.
A estrutura exigida possui as colunas: `sintoma_1 | sintoma_2 | doenca_associada`.


In [ ]:
# Carregando e visualizando o mapa de conhecimento
df_mapa = pd.read_csv(ARQUIVO_MAPA)
print(f"Total de regras no mapa de conhecimento: {len(df_mapa)}")
print(f"Doenças distintas mapeadas: {df_mapa['doenca_associada'].nunique()}")
display(df_mapa.head(15))


## 🧠 5. Motor de Inferência e Extração de Sintomas
O algoritmo aplica:
1. **Normalização de texto:** minúsculas, remoção de caracteres espúrios.
2. **Match Completo (Alta Confiança 🔴):** Ambos os sintomas (`sintoma_1` e `sintoma_2`) são identificados no relato.
3. **Match Parcial (Confiança Moderada 🟡):** Apenas um dos sintomas chave é identificado.


In [ ]:
def normalizar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\sáàâãéèêíìîóòôõúùûçÁÀÂÃÉÈÊÍÌÎÓÒÔÕÚÙÛÇ]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def diagnosticar(frase, regras):
    frase_norm = normalizar(frase)
    diagnosticos = []
    doencas_vistas = set()

    # Match Completo (ambos os sintomas)
    for _, regra in regras.iterrows():
        s1 = normalizar(regra["sintoma_1"])
        s2 = normalizar(regra["sintoma_2"])
        doenca = regra["doenca_associada"]

        if s1 in frase_norm and s2 in frase_norm:
            if doenca not in doencas_vistas:
                diagnosticos.append({
                    "doenca": doenca,
                    "sintomas": [regra["sintoma_1"], regra["sintoma_2"]],
                    "confianca": "ALTA",
                    "tipo": "Match Completo"
                })
                doencas_vistas.add(doenca)

    # Match Parcial (apenas 1 sintoma)
    for _, regra in regras.iterrows():
        s1 = normalizar(regra["sintoma_1"])
        s2 = normalizar(regra["sintoma_2"])
        doenca = regra["doenca_associada"]

        if (s1 in frase_norm or s2 in frase_norm) and doenca not in doencas_vistas:
            sintoma_achado = regra["sintoma_1"] if s1 in frase_norm else regra["sintoma_2"]
            diagnosticos.append({
                "doenca": doenca,
                "sintomas": [sintoma_achado],
                "confianca": "MODERADA",
                "tipo": "Match Parcial"
            })
            doencas_vistas.add(doenca)

    return diagnosticos


## 🩺 6. Execução e Apresentação dos Diagnósticos por Paciente


In [ ]:
for idx, relato in enumerate(frases_pacientes, 1):
    resultados = diagnosticar(relato, df_mapa)
    print(f"{'='*80}")
    print(f"PACIENTE #{idx:02d}")
    print(f"Relato: "{relato}"")
    print(f"{'-'*80}")
    if resultados:
        for diag in sorted(resultados, key=lambda x: 0 if x['confianca'] == 'ALTA' else 1):
            icone = '🔴 [ALTA CONFIANÇA]' if diag['confianca'] == 'ALTA' else '🟡 [MODERADA]'
            print(f"  {icone} {diag['doenca']}")
            print(f"     Sintomas Chave: {', '.join(diag['sintomas'])} ({diag['tipo']})")
    else:
        print("  ⚪ Nenhuma correlação direta identificada. Recomenda-se avaliação clínica geral.")
    print()


## 📊 7. Conclusões e Governança da Solução
1. **Desempenho:** Todas as 10 frases foram mapeadas com sucesso para diagnósticos de **Alta Confiança**, cobrindo emergências agudas (Infarto, Crise Hipertensiva) e condições crônicas (Insuficiência Cardíaca Congestiva, Nefropatia).
2. **Escalabilidade:** O motor baseado em regras atua como uma ontologia determinística rápida e auditável.
3. **Evolução:** Em ambientes produtivos, essa camada simbólica é combinada com modelos probabilísticos e embeddings de linguagem biomédica (como BioBERT/ClinicalBERT) para tolerar variações coloquiais mais amplas.
